In [1]:
from ctlearn_manager import CTLearnTriModelManager, load_model_from_index
import pandas as pd
import numpy as np

# 💾 Load models

In [2]:
def get_surrounding_nodes(index, nodes, num_surrounding=1):
    surrounding_indices = []
    for i in range(1, num_surrounding + 1):
        if index - i >= 0:
            surrounding_indices.append(index - i)
        if index + i < len(nodes):
            surrounding_indices.append(index + i)
        if len(surrounding_indices) >= num_surrounding:
            break
    return surrounding_indices

def angular_distance(ze1, az1, ze2, az2):
    ze1, az1, ze2, az2 = map(np.radians, [ze1, az1, ze2, az2])
    delta_az = az2 - az1
    delta_ze = ze2 - ze1
    a = np.sin(delta_ze / 2)**2 + np.cos(ze1) * np.cos(ze2) * np.sin(delta_az / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return c

def get_testing_node(first_node_index, training_nodes_gammas, training_nodes_protons, testing_nodes_gammas):
    zes = [training_nodes_gammas['ze'][first_node_index], training_nodes_gammas['ze'][first_node_index + 1]]
    azs = [training_nodes_gammas['az'][first_node_index], training_nodes_gammas['az'][first_node_index + 1]]
    zes_protons = []
    azs_protons = []
    for index in [first_node_index, first_node_index + 1]:
        surrounding_indices = get_surrounding_nodes(index, training_nodes_protons['ze'])
        zes_protons.extend(training_nodes_protons['ze'][surrounding_indices])
        azs_protons.extend(training_nodes_protons['az'][surrounding_indices])
    same_ze_indices = training_nodes_protons[training_nodes_protons['ze'].isin(zes_protons)].index
    zes_protons = training_nodes_protons['ze'][same_ze_indices]
    azs_protons = training_nodes_protons['az'][same_ze_indices]
    # Find the closest gamma point testing node for the two training gamma diffuse nodes
    min_distance = float('inf')
    closest_testing_node = None
    mid_ze = np.mean(zes)
    mid_az = np.mean(azs)

    distances = angular_distance(mid_ze, mid_az, testing_nodes_gammas['ze'], testing_nodes_gammas['az'])
    closest_index = np.argmin(distances)
    closest_testing_node = (testing_nodes_gammas['ze'][closest_index], testing_nodes_gammas['az'][closest_index])

    # Extract the zenith and azimuth angles of the closest testing node
    closest_ze_testing = [closest_testing_node[0]]
    closest_az_testing = [closest_testing_node[1]]
    return zes, azs, zes_protons, azs_protons, closest_ze_testing, closest_az_testing

In [3]:
MODEL_INDEX_FILE = "/users/blacave/PhD/LST/CTLearnManager/ctlearn_models_index.h5"
training_nodes_gammas = pd.read_csv('/users/blacave/PhD/LST/CTLearnManager/TrainingNodesGammaDiffuse.csv')
training_nodes_protons = pd.read_csv('/users/blacave/PhD/LST/CTLearnManager/TrainingNodesProtonDiffuse.csv')
testing_nodes_gammas = pd.read_csv('/users/blacave/PhD/LST/CTLearnManager/TestingNodesGammaPoint.csv')
for i in range(10):
    energy_model = load_model_from_index(f"LST1_energy_CRABdec_{i}", MODEL_INDEX_FILE)
    direction_model = load_model_from_index(f"LST1_direction_CRABdec_{i}", MODEL_INDEX_FILE)
    type_model = load_model_from_index(f"LST1_type_CRABdec_{i}", MODEL_INDEX_FILE)
    Stereo_Tri_Model = CTLearnTriModelManager(direction_model=direction_model, energy_model=energy_model, type_model=type_model)

    zes, azs, zes_protons, azs_protons, closest_ze_testing, closest_az_testing = get_testing_node(i, training_nodes_gammas, training_nodes_protons, testing_nodes_gammas) 
    Stereo_Tri_Model.set_testing_directories(
        testing_gamma_dirs = ["/capstor/scratch/cscs/tmiener/datasets/LST1/test/gamma/"], 
        testing_gamma_zenith_distances = closest_ze_testing, 
        testing_gamma_azimuths = closest_az_testing, 
        testing_gamma_patterns = [f"gamma_theta_{int(closest_ze_testing[0]) if closest_ze_testing[0].is_integer() else closest_ze_testing[0]}_az_{int(closest_az_testing[0]) if closest_az_testing[0].is_integer() else closest_az_testing[0]}_runs*.dl1.h5"],
        # testing_proton_dirs = ["/home/blacave/CTLearn/Data/DL1/SST1M/MC/Proton_diffuse/20deg/merged/testing/"], 
        # testing_proton_zenith_distances = [closest_ze_testing], 
        # testing_proton_azimuths = [closest_az_testing]
        # testing_proton_patterns = ["proton*.h5"]
        )

🧠 Model name: LST1_energy_CRABdec_0
🧠 Model name: LST1_direction_CRABdec_0
🧠 Model name: LST1_type_CRABdec_0
💾 Model LST1_direction_CRABdec_0 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_energy_CRABdec_0 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_type_CRABdec_0 testing data update:
	➡️ Testing gamma data updated
🧠 Model name: LST1_energy_CRABdec_1
🧠 Model name: LST1_direction_CRABdec_1
🧠 Model name: LST1_type_CRABdec_1
💾 Model LST1_direction_CRABdec_1 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_energy_CRABdec_1 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_type_CRABdec_1 testing data update:
	➡️ Testing gamma data updated
🧠 Model name: LST1_energy_CRABdec_2
🧠 Model name: LST1_direction_CRABdec_2
🧠 Model name: LST1_type_CRABdec_2
💾 Model LST1_direction_CRABdec_2 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1_energy_CRABdec_2 testing data update:
	➡️ Testing gamma data updated
💾 Model LST1

# 🗃️ Set testing files and directions

In [4]:
# Stereo_Tri_Model.set_testing_directories(
#     testing_gamma_dirs = ["/home/blacave/CTLearn/Data/DL1/SST1M/MC/Gamma_point/20deg/testing/"], 
#     testing_proton_dirs = ["/home/blacave/CTLearn/Data/DL1/SST1M/MC/Proton_diffuse/20deg/merged/testing/"], 
#     testing_gamma_zenith_distances = [20], 
#     testing_gamma_azimuths = [0], 
#     testing_proton_zenith_distances = [20], 
#     testing_proton_azimuths = [0]
#     )

# 🧪 Launch testing

In [5]:
Stereo_Tri_Model.get_available_testing_directions()

(ZD, Az): (37.814, 90.0)


In [6]:
Stereo_Tri_Model.launch_testing(37.814, 90, 
    ["/capstor/scratch/cscs/blacave/Testing_models/"], 
    launch_particle_type='gamma', 
    sbatch_scripts_dir="/users/blacave/PhD/LST/CTLearnManager/sbatch/",
    cluster='cscs', account='cta04', python_env='ctlearn-cluster'
    )

💾 Model LST1_direction_CRABdec_9 DL2 data update:
	➡️ Testing DL2 gamma data updated
💾 Model LST1_energy_CRABdec_9 DL2 data update:
	➡️ Testing DL2 gamma data updated
💾 Model LST1_type_CRABdec_9 DL2 data update:
	➡️ Testing DL2 gamma data updated
💾 Testing script saved in /users/blacave/PhD/LST/CTLearnManager/sbatch//gamma_theta_37.814_az_90_runs1-500.dl1.sh
Submitted batch job 190551


In [3]:
Stereo_Tri_Model.merge_DL2_files(20, 0, "/home/blacave/CTLearn/Data/DL2/Testing/merged/gamma_point_50_300E3GeV_20_20deg.h5", "/home/blacave/CTLearn/Data/DL2/Testing/merged/proton_diffuse_400_500E3GeV_20_20deg.h5", overwrite=True)

Merging: 100%|██████████| 3/3 [00:14<00:00,  4.96s/Files]


💾 Model direction_stereo_20deg DL2 merged data update:
	➡️ Testing DL2 gamma merged data updated
💾 Model energy_stereo_20deg DL2 merged data update:
	➡️ Testing DL2 gamma merged data updated
💾 Model type_stereo_20deg DL2 merged data update:
	➡️ Testing DL2 gamma merged data updated


Merging: 100%|██████████| 6/6 [01:23<00:00, 13.85s/Files]


💾 Model direction_stereo_20deg DL2 merged data update:
	➡️ Testing DL2 proton merged data updated
💾 Model energy_stereo_20deg DL2 merged data update:
	➡️ Testing DL2 proton merged data updated
💾 Model type_stereo_20deg DL2 merged data update:
	➡️ Testing DL2 proton merged data updated
